In [7]:
from dotenv import load_dotenv
import os
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

True

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
DIMENSIONS = 384

In [14]:
cluster_endpoint = os.environ["MILVUS_ENDPOINT"]
token = os.environ["MILVUS_TOKEN"]

In [15]:
from langchain_milvus import Milvus

In [16]:
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="first_collection",
    collection_description="collection from local",
    connection_args={"uri": cluster_endpoint, "token": token},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
)

### Ingestion to store

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=100, is_separator_regex=True
)
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_72026/3464138467.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [19]:
pdf_path = (
    "/home/gaurav/Documents/advanced_rag/data/EthanolBlendingInIndia_compressed.pdf"
)
loader = PyPDFLoader(file_path=pdf_path)

In [20]:
docs = loader.load_and_split(text_splitter=splitter)


In [23]:
# vector_store.add_documents(documents=docs)

### Retriever

In [24]:
retriever = vector_store.as_retriever()

### LLM Groq

In [28]:
load_dotenv()
llm_key = os.environ["GROQ_KEY"]

In [31]:
from langchain_groq import ChatGroq

In [34]:
llm = ChatGroq(api_key=llm_key, model="openai/gpt-oss-20b")

## Prompt

In [36]:
from langchain_core.prompts import ChatPromptTemplate

In [38]:
prompt_text = """
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Instructions:
- If the answer is not present in the context, say "I don't know based on the provided context."
- Do not invent facts.
- Keep the answer clear and concise.

Answer:
"""

In [39]:
prompt_template = ChatPromptTemplate([("system", prompt_text), ("human", "{question}")])

In [41]:
chain = retriever | prompt_template | llm

In [44]:
from typing import TypedDict

from langchain_core.documents import Document
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END


# 1. Define the graph state
class RAGState(TypedDict):
    question: str
    context: list[Document]
    answer: BaseMessage


# 2. Retrieve relevant documents from Milvus
def retrieve(state: RAGState) -> dict:
    documents = retriever.invoke(state["question"])
    return {"context": documents}


# 3. Generate the answer using Groq
def generate(state: RAGState) -> dict:
    context = "\n\n".join(doc.page_content for doc in state["context"])

    response = (prompt_template | llm).invoke(
        {
            "context": context,
            "question": state["question"],
        }
    )

    return {"answer": response}


# 4. Build the LangGraph workflow
builder = StateGraph(RAGState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

rag_graph = builder.compile()


In [48]:
# 5. Run the RAG pipeline
result = rag_graph.invoke({"question": "what is fibonanci series"})

print(result["answer"].content)

I don't know based on the provided context.
